# 03 · Simulación Monte Carlo

**Fase CRISP-DM:** 2. Entendimiento de los datos

**Objetivo:** generar un año de comportamiento de los comensales registrados de Puzzle en la CDMX: quiénes son, cuánto gastan, qué les gusta, a dónde van, con quién y qué piden.

| Entradas | Salidas (`data/synthetic/<run_id>/`) |
|---|---|
| Menús del nb 02, AGEB con población (nb 01), ENIGH 2024, Censo 2020 | `diners`, `diner_profiles`, `dining_sessions`, `guests`, `order_items`, `dish_ratings`, `visit_ratings` + `ground_truth/` |

**La idea central: una verdad oculta.** Primero se sortean los gustos latentes de cada comensal y **después** se simula su comportamiento a partir de ellos. Así, al evaluar el modelo se puede medir si **recuperó** esos gustos (notebooks 14 y 15). La verdad nunca entra al contrato (ADR 0002).

## Configuración

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd

from puzzle_recsys.catalog.library import load_library
from puzzle_recsys.config import CONFIG_DIR, load_paths, load_simulation
from puzzle_recsys.repository.parquet_repo import run_id_for, start_run
from puzzle_recsys.simulation.assemble import assert_no_truth_leak, split_truth, tastes_frame
from puzzle_recsys.simulation.availability import attach_availability, regenerate_availability
from puzzle_recsys.simulation.distributions import make_rng
from puzzle_recsys.simulation.engine import build_world_index, haversine_km, simulate_visits
from puzzle_recsys.simulation.feedback import simulate_ratings
from puzzle_recsys.simulation.noise import apply_noise
from puzzle_recsys.simulation.population import (
    onboarding_answers,
    sample_budgets,
    sample_homes,
    sample_tastes,
)
from puzzle_recsys.sources.download import download_source
from puzzle_recsys.sources.enigh import read_enigh_households
from puzzle_recsys.sources.inegi import read_ageb_indicators

paths, sim = load_paths(), load_simulation()
repo = start_run(
    paths.synthetic,
    run_id_for(sim.run_name, sim.seed),
    config_files=[CONFIG_DIR / "simulation.yaml"],
)
library = load_library(paths.external / "canonical_dishes.yaml")
names = (
    "restaurants",
    "canonical_dishes",
    "menu_items",
    "menu_item_attributes",
    "menu_item_price_history",
    "menu_item_availability_events",
)
tables = {name: repo.read_table(name) for name in names}
tables["ground_truth_menu_items"] = repo.ground_truth.read_table("menu_items")
n = sim.scale.n_diners
print(
    f"Comensales: {n:,} · restaurantes: {len(tables['restaurants']):,} · meses: {sim.scale.months}"
)

Comensales: 20,000 · restaurantes: 1,500 · meses: 12


## 1. Dónde viven

**Muestreo con probabilidad proporcional a la población** de cada AGEB (Censo 2020) y un punto uniforme dentro del polígono (muestreo por rechazo). La casa **no** se guarda en el contrato: Puzzle no pide la dirección del comensal; solo se usa para simular a dónde va.

In [2]:
ageb = gpd.read_parquet(paths.interim / "world" / "ageb.parquet")
homes = sample_homes(ageb, n, make_rng(sim.seed, "homes"))
homes.merge(ageb[["cvegeo", "alcaldia"]], on="cvegeo")["alcaldia"].value_counts(
    normalize=True
).round(3).to_frame("% de comensales").T

alcaldia,Iztapalapa,Gustavo A. Madero,Álvaro Obregón,Tlalpan,Coyoacán,Cuauhtémoc,Venustiano Carranza,Benito Juárez,Azcapotzalco,Xochimilco,Miguel Hidalgo,Tláhuac,Iztacalco,La Magdalena Contreras,Cuajimalpa de Morelos,Milpa Alta
% de comensales,0.199,0.131,0.082,0.072,0.07,0.061,0.051,0.05,0.047,0.046,0.045,0.043,0.042,0.026,0.023,0.013


## 2. Cuánto gasta por visita

**Generación condicional + bootstrap suavizado:** el percentil socioeconómico del comensal (escolaridad de su AGEB) elige su decil de ingreso en la ENIGH 2024; dentro de ese decil (±1) se remuestrea un hogar **ponderado por el factor de expansión** y se suaviza con un kernel lognormal. Gasto trimestral por persona ÷ salidas por trimestre (supuesto S2).

In [3]:
households = read_enigh_households(download_source("enigh"), entity="09")
ses = read_ageb_indicators(paths.raw / "ageb_mza_urbana_09_cpv2020_csv.zip", ["GRAPROES"])
graproes = homes["cvegeo"].map(ses.set_index("cvegeo")["graproes"])
ses_pct = graproes.fillna(graproes.median()).rank(pct=True).to_numpy()
budgets, frequency = sample_budgets(
    ses_pct, households, sim.population["budget"], make_rng(sim.seed, "budgets")
)
eats_out = (households["food_away_pc"] > 0).mean()
print(f"Hogares ENIGH CDMX: {len(households):,} · con gasto fuera > 0: {eats_out:.0%}")
pd.DataFrame({"presupuesto por visita (MXN)": budgets, "frecuencia relativa": frequency}).groupby(
    pd.qcut(ses_pct, 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
).median().round(2).T

Hogares ENIGH CDMX: 2,576 · con gasto fuera > 0: 68%


,Q1,Q2,Q3,Q4,Q5
presupuesto por visita (MXN),233.85,280.83,344.35,422.06,531.62
frecuencia relativa,0.56,0.74,1.01,1.37,1.93


## 3. Qué le gusta (la verdad oculta)

- **Cocina:** Dirichlet centrada en la oferta real de la ciudad (la gente come lo que hay) con concentración baja: cada quien tiene pocas cocinas favoritas.
- **Proteína:** Dirichlet. **Ingredientes:** algunos le encantan, otros los evita.
- **Picante:** tolerancia 0–3. **Precio y distancia:** sensibilidades **lognormales propias de cada comensal** (logit mixto).
- **Hábitos** por tipo de plato y rasgo **"compartidor"**: Beta. **Dieta y alergias:** prevalencias configuradas (supuesto S10).
- **Onboarding:** versión ruidosa e incompleta de lo anterior (MAR); nunca inventa alergias.

In [4]:
restaurants = tables["restaurants"]
cuisines = sorted(restaurants["cuisine_pred"].unique())
share = restaurants["cuisine_pred"].value_counts(normalize=True).reindex(cuisines).to_numpy()
proteins = ["BEEF", "PORK", "CHICKEN", "FISH", "SEAFOOD", "PLANT", "EGG", "CHEESE", "MIXED"]
taste_kwargs = dict(
    cuisines=cuisines,
    cuisine_share=share,
    proteins=proteins,
    ingredients=list(library.ingredients),
    allergens=list(library.allergens),
    cfg=sim.population,
)
tastes = sample_tastes(n, rng=make_rng(sim.seed, "tastes"), **taste_kwargs)
pool = sim.visits["anonymous_pool_size"]
anonymous = (
    sample_tastes(pool, rng=make_rng(sim.seed, "anonymous"), **taste_kwargs),
    sample_budgets(
        make_rng(sim.seed, "anon_ses").random(pool),
        households,
        sim.population["budget"],
        make_rng(sim.seed, "anon_budgets"),
    )[0],
)
answers = onboarding_answers(tastes, sim.population["onboarding"], make_rng(sim.seed, "onboarding"))
summary = {
    "cocinas con afinidad > 20 % (mediana)": float(
        np.median((tastes.cuisine_affinity > 0.2).sum(axis=1))
    ),
    "vegetarianos o veganos": float((tastes.diet != "ninguna").mean()),
    "con al menos una alergia": float(tastes.allergies.any(axis=1).mean()),
    "completó el onboarding": float(answers["completed"].mean()),
}
pd.Series(summary).round(3).to_frame("valor").T

,cocinas con afinidad > 20 % (mediana),vegetarianos o veganos,con al menos una alergia,completó el onboarding
valor,2.0,0.077,0.101,0.604


## 4. Índice del mundo y agotados según la demanda

Se precalculan atributos, vigencias y precios de cada platillo, y se sortea su **atractivo latente** (cola larga, `-α·log(rango)`) y la **calidad de ejecución** de cada restaurante (Laplace, colas pesadas).

Los agotados exógenos del nb 02 se reemplazan por **agotados según la demanda latente**: se agota más lo que más se pide. La demanda latente representa a **toda** la clientela del restaurante, no solo a los usuarios simulados, y no depende de las visitas (no hay circularidad).

In [5]:
tz = sim.calendar["timezone"]
habit_beta = sim.population["taste"]["course_habit_beta"]
course_habit_means = {"MAIN": sim.visits["main_course_rate"]} | {
    c: b["a"] / (b["a"] + b["b"]) for c, b in habit_beta.items()
}
index, canonical_appeal = build_world_index(
    tables,
    library,
    cuisines,
    sim.choice,
    tz,
    make_rng(sim.seed, "world_index"),
    course_habit_means=course_habit_means,
)
end = pd.Timestamp(sim.calendar["start_date"], tz=tz) + pd.DateOffset(months=sim.scale.months)
availability = regenerate_availability(
    index,
    tables["menu_items"],
    sim.supply["evolution"]["daily_sold_out_rate"],
    sim.supply["evolution"]["max_daily_sold_out_rate"],
    end,
    make_rng(sim.seed, "availability"),
)
attach_availability(index, availability, tz)
availability["reason"].value_counts().to_frame("eventos").T

reason,SOLD_OUT,RESTOCKED,CREATED,ARCHIVED
eventos,198241,198241,36115,6265


## 5. Monte Carlo: un año de visitas

Para cada comensal: tiempos entre visitas **Weibull** (escala propia: pocos muy frecuentes), *thinning* por día de la semana, hora del perfil empírico → restaurante por **logit mixto** dentro de su conjunto de consideración (K más cercanos + los que frecuenta) → mesa (`pax`, acompañantes registrados o anónimos) → pedidos por tipo de plato entre lo **disponible a esa hora**, con alergias y dieta como **filtros duros** → platillo para compartir con utilidad de grupo (promedio + miseria mínima).

In [6]:
cfg = {
    k: getattr(sim, k) for k in ("visits", "choice", "sharing", "population", "ratings", "noise")
}
out = simulate_visits(
    index,
    tastes,
    homes,
    budgets,
    anonymous,
    cfg,
    sim.calendar,
    months=sim.scale.months,
    rng=make_rng(sim.seed, "visits"),
    frequency=frequency,
)
sizes = {k: len(out[k]) for k in ("dining_sessions", "guests", "order_items")}
print("{dining_sessions:,} visitas · {guests:,} invitados · {order_items:,} líneas".format(**sizes))

512,907 visitas · 1,473,303 invitados · 3,396,760 líneas


## 6. Calificaciones (MNAR) y ruido de datos

In [7]:
ratings = simulate_ratings(
    out["order_items"],
    out["guests"],
    out["dining_sessions"],
    sim.ratings,
    make_rng(sim.seed, "ratings"),
)
diner_ids = [f"d{i:06d}" for i in range(n)]
registered_at = pd.Timestamp(sim.calendar["start_date"], tz=tz) - pd.to_timedelta(
    make_rng(sim.seed, "registration").integers(1, 365, n), unit="D"
)
diners = pd.DataFrame({"diner_id": diner_ids, "registered_at": registered_at.tz_convert("UTC")})
order_items, guests, truth_lines, truth_guests = split_truth(out["order_items"], out["guests"])
contract, noise_truth = apply_noise(
    {"order_items": order_items, "guests": guests, "diners": diners},
    sim.noise,
    make_rng(sim.seed, "noise"),
)
display(
    ratings["dish_ratings"]["stars"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
    .to_frame("% estrellas")
    .T
)
pd.Series(
    {
        "líneas sin comensal (mesero)": contract["order_items"]["guest_id"].isna().mean(),
        "cuentas duplicadas": len(noise_truth["duplicate_accounts"]),
    }
).round(3).to_frame("valor").T

stars,1,2,3,4,5
% estrellas,0.122,0.091,0.21,0.247,0.33


,líneas sin comensal (mesero),cuentas duplicadas
valor,0.04,204.0


## 7. ¿La verdad oculta mueve el comportamiento?

Antes de guardar se comprueba que los gustos latentes **sí** se reflejan en lo observado. Si no, el modelo no tendría nada que aprender.

In [8]:
lines = truth_lines.merge(
    out["order_items"][["order_item_id", "guest_id", "session_id"]], on="order_item_id"
)
host = out["guests"][out["guests"]["is_host"]][["session_id", "diner_id"]]
visits = out["dining_sessions"].merge(host, on="session_id")
visits["i"] = visits["diner_id"].str[1:].astype(int)
r_pos = {r: k for k, r in enumerate(index.rest_ids)}
home_rad = np.radians(homes[["lat", "lng"]].to_numpy())
visits["km"] = [
    float(haversine_km(home_rad[i], index.rest_latlng[[r_pos[r]]])[0])
    for i, r in zip(visits["i"], visits["restaurant_id"], strict=True)
]
visits["beta_q"] = pd.qcut(
    tastes.distance_sensitivity[visits["i"]],
    4,
    labels=["β Q1 (le da igual)", "Q2", "Q3", "β Q4 (odia moverse)"],
)
cuisine_of = restaurants.set_index("restaurant_id")["cuisine_pred"]
fav = np.array(cuisines)[tastes.cuisine_affinity.argmax(axis=1)]
visits["fav_cuisine"] = cuisine_of.reindex(visits["restaurant_id"]).to_numpy() == fav[visits["i"]]
expected = (share[tastes.cuisine_affinity.argmax(axis=1)]).mean()
display(
    visits.groupby("beta_q", observed=True)["km"]
    .median()
    .round(2)
    .to_frame("km mediana al restaurante")
    .T
)
pd.Series(
    {
        "visitas a su cocina favorita (observado)": visits["fav_cuisine"].mean(),
        "si eligiera al azar según la oferta": expected,
    }
).round(3).to_frame("proporción").T

beta_q,β Q1 (le da igual),Q2,Q3,β Q4 (odia moverse)
km mediana al restaurante,1.61,1.5,1.43,1.31


,visitas a su cocina favorita (observado),si eligiera al azar según la oferta
proporción,0.471,0.194


### Correcciones verificadas

Tras la primera corrida se encontraron y corrigieron ocho problemas. Aquí se comprueba que cada corrección funcionó:

1. **El presupuesto no influía en qué platillo se elegía** (se cancelaba en el logit). Ahora el precio entra como distancia a un **precio ideal anclado al menú** (anclaje: en un restaurante caro se pide algo "normal para ese lugar"). → El gasto real debe crecer con el nivel socioeconómico.
2. **Presupuesto inflado en el quintil alto.** Ahora el gasto se descompone en frecuencia × ticket. → Quien gasta más sale más seguido **y** gasta algo más por visita.
3. **Perfil semanal deformado** por el *thinning*. → Lo observado debe coincidir con lo configurado.
4. **Acompañantes que vivían a 14 km.** Ahora son vecinos (KNN). → La distancia entre casas debe bajar mucho.
5. **Ocasiones especiales:** en un aniversario se va a un lugar más caro y se gasta más.
6. **Una persona en dos mesas a la vez** (encontrado en el EDA del notebook 06): un acompañante registrado podía sentarse en otra mesa minutos después, o salir dos veces en la misma mesa. Ahora una **agenda global** impide estar en dos mesas a menos de 3 h (S15). → Se verifica en el notebook 05 (consistencia).
7. **La hora no influía en nada** (encontrado en el EDA bivariado del notebook 07): en el desayuno se pedía lo mismo que en la cena. Ahora el **contexto** cambia la cocina elegida, los tipos de plato, algunos platillos (desayuno, alcohol) y el tamaño de mesa (S16). → Se verifica en el notebook 07.
8. **El gusto por compartir no cambiaba si la mesa compartía** (encontrado en el EDA multivariado del notebook 08): el rasgo `sharer` solo decidía quién pedía el plato al centro. Ahora la probabilidad de compartir sube con el gusto medio de la mesa, normalizado para que una mesa promedio conserve la tasa de siempre (S17). → Se verifica en el notebook 08.


In [9]:
truth_v = out["ground_truth_visits"].merge(out["dining_sessions"], on="session_id")
truth_v["i"] = truth_v["host_diner_id"].str[1:].astype(int)
host_lines = out["order_items"][~out["order_items"]["is_shared"]].merge(
    out["guests"][out["guests"]["is_host"]][["guest_id", "session_id"]],
    on=["guest_id", "session_id"],
)
spent = host_lines.groupby("session_id")["unit_price_cents"].sum() / 100
truth_v["gasto"] = truth_v["session_id"].map(spent)
truth_v["nivel"] = truth_v["restaurant_id"].map(
    dict(zip(index.rest_ids, index.rest_price_level, strict=True))
)
quintil = pd.qcut(ses_pct, 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
truth_v["quintil"] = np.asarray(quintil)[truth_v["i"]]
visitas_por_comensal = truth_v.groupby("i").size().reindex(range(n), fill_value=0)

display(
    pd.DataFrame(
        {
            "gasto mediano del anfitrión por visita": truth_v.groupby("quintil", observed=True)[
                "gasto"
            ].median(),
            "visitas por comensal (mediana)": visitas_por_comensal.groupby(
                np.asarray(quintil)
            ).median(),
        }
    )
    .round(0)
    .T
)

weekday = out["dining_sessions"]["opened_at"].dt.tz_convert(tz).dt.day_of_week
w = np.asarray(sim.visits["weekday_weights"])
display(
    pd.DataFrame(
        {
            "configurado": w / w.sum(),
            "observado": weekday.value_counts(normalize=True).sort_index().to_numpy(),
        },
        index=["lun", "mar", "mié", "jue", "vie", "sáb", "dom"],
    )
    .round(3)
    .T
)

g_reg = out["guests"][~out["guests"]["is_host"] & out["guests"]["taste_ref"].str.startswith("d")]
g_reg = g_reg.merge(
    out["guests"][out["guests"]["is_host"]][["session_id", "taste_ref"]],
    on="session_id",
    suffixes=("", "_host"),
)
a = home_rad[g_reg["taste_ref_host"].str[1:].astype(int)]
b = home_rad[g_reg["taste_ref"].str[1:].astype(int)]
km_amigos = (
    2
    * 6371
    * np.arcsin(
        np.sqrt(
            np.sin((b[:, 0] - a[:, 0]) / 2) ** 2
            + np.cos(a[:, 0]) * np.cos(b[:, 0]) * np.sin((b[:, 1] - a[:, 1]) / 2) ** 2
        )
    )
)
sold = availability[availability["reason"] == "SOLD_OUT"].groupby("menu_item_id").size()
pd.Series(
    {
        "km entre casas de amigos (mediana; antes 14.0)": float(np.median(km_amigos)),
        "% visitas que son ocasión especial": truth_v["occasion"].mean() * 100,
        "nivel de precio del lugar: ocasión / normal": truth_v.groupby("occasion")["nivel"]
        .median()
        .pipe(lambda s: s[True] / s[False]),
        "gasto del anfitrión: ocasión / normal": truth_v.groupby("occasion")["gasto"]
        .median()
        .pipe(lambda s: s[True] / s[False]),
        "máx. días agotado al año (antes 106)": int(sold.max()),
    }
).round(2).to_frame("valor").T

,Q1,Q2,Q3,Q4,Q5
gasto mediano del anfitrión por visita,185.0,195.0,209.0,219.0,240.0
visitas por comensal (mediana),11.0,14.0,19.0,25.0,36.0


,lun,mar,mié,jue,vie,sáb,dom
configurado,0.100,0.110,0.12,0.140,0.190,0.2,0.14
observado,0.102,0.109,0.12,0.141,0.189,0.2,0.14


,km entre casas de amigos (mediana; antes 14.0),% visitas que son ocasión especial,nivel de precio del lugar: ocasión / normal,gasto del anfitrión: ocasión / normal,máx. días agotado al año (antes 106)
valor,1.0,6.9,1.05,1.05,67.0


## 8. Verificaciones de integridad y guardado

Se verifican en los datos completos las reglas que nunca deben romperse, y que **ninguna columna de verdad** llegue al contrato.

In [10]:
pos = {item: k for k, item in enumerate(index.item_ids)}
refs = out["guests"].set_index("guest_id")["taste_ref"]
own = out["order_items"][~out["order_items"]["is_shared"]]
k_items = own["menu_item_id"].map(pos).to_numpy()
ref = own["guest_id"].map(refs)
registered = ref.str.startswith("d").to_numpy()
rows = ref[registered].str[1:].astype(int).to_numpy()
allergy_hits = (
    (index.items.allergens[k_items[registered]] & tastes.allergies[rows]).any(axis=1).sum()
)
veg = tastes.diet[rows] != "ninguna"
veg_hits = (~index.items.vegetarian[k_items[registered]][veg]).sum()
pax_ok = (
    out["guests"]
    .groupby("session_id")
    .size()
    .reindex(out["dining_sessions"]["session_id"])
    .to_numpy()
    == out["dining_sessions"]["pax"].to_numpy()
).all()
checks = {
    "alérgenos servidos a un alérgico": int(allergy_hits),
    "carne servida a un vegetariano/vegano": int(veg_hits),
    "mesas con pax ≠ invitados": int(not pax_ok),
}
assert all(v == 0 for v in checks.values()), checks
pd.Series(checks, name="violaciones").to_frame().T

,alérgenos servidos a un alérgico,carne servida a un vegetariano/vegano,mesas con pax ≠ invitados
violaciones,0,0,0


In [11]:
final = {
    "diners": contract["diners"],
    "diner_profiles": answers.assign(diner_id=diner_ids),
    "dining_sessions": out["dining_sessions"],
    "guests": contract["guests"],
    "order_items": contract["order_items"],
    "dish_ratings": ratings["dish_ratings"],
    "visit_ratings": ratings["visit_ratings"],
    "menu_item_availability_events": availability,
}
assert_no_truth_leak(final)
for name, table in final.items():
    repo.write_table(name, table)
truth = {
    "diner_tastes": tastes_frame(tastes, diner_ids),
    "diner_homes": homes.assign(
        diner_id=diner_ids, budget=budgets, frequency=frequency, ses_percentile=ses_pct
    ),
    "anonymous_tastes": tastes_frame(anonymous[0], [f"a{j:05d}" for j in range(pool)]).assign(
        budget=anonymous[1]
    ),
    "canonical_appeal": canonical_appeal,
    "menu_item_quality": pd.DataFrame(
        {"menu_item_id": index.item_ids, "quality": index.items.quality}
    ),
    "visits": out["ground_truth_visits"],
    "order_item_utilities": truth_lines,
    "guest_tastes": truth_guests,
    **noise_truth,
}
for name, table in truth.items():
    repo.ground_truth.write_table(name, table)
pd.Series({n_: len(t) for n_, t in final.items()}, name="filas").to_frame().T

,diners,diner_profiles,dining_sessions,guests,order_items,dish_ratings,visit_ratings,menu_item_availability_events
filas,20204,20000,512907,1473303,3396760,419941,304192,438862


## Conclusiones

- Población anclada a datos reales: **dónde vive** (Censo 2020 por AGEB) y **cuánto gasta** (ENIGH 2024, condicional al nivel socioeconómico y ponderada por el factor de expansión).
- La **verdad oculta mueve el comportamiento**: quienes son más sensibles a la distancia van a restaurantes más cercanos, y la gente visita su cocina favorita mucho más que al azar. Hay señal que aprender.
- **Filtros duros verificados** sobre todo el año: ningún alérgico recibió su alérgeno y ningún vegetariano recibió carne.
- **Precio realista:** el precio ideal combina el presupuesto con lo típico del menú (anclaje), y las **ocasiones especiales** llevan a lugares más caros. El gasto se descompone en frecuencia × ticket.
- **Faltantes y ruido conocidos** (onboarding incompleto MCAR, calificaciones MNAR, pedidos sin comensal, cuentas duplicadas): dan algo real que diagnosticar en el notebook 05.
- **Pasa al notebook 04:** validar que la simulación se parezca a la realidad (hechos estilizados) y calibrar la relación señal/ruido.